<a href="https://colab.research.google.com/github/vallari24/zero-to-research/blob/main/notebooks/07_gpt_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 GPT From Scratch: Colab Setup

This notebook prepares the repo for following Karpathy's GPT lecture with TinyStories instead of Tiny Shakespeare.

It creates a local `input.txt` at the repo root because the lecture code expects that exact filename.

In Colab, set `Runtime -> Change runtime type -> GPU` before running the training cells.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/vallari24/zero-to-research.git"
REPO_DIR = Path("/content/zero-to-research")

def run(cmd):
    print("+", " ".join(str(part) for part in cmd))
    subprocess.run(cmd, check=True)

if Path("/content").exists():
    if not REPO_DIR.exists():
        run(["git", "clone", REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
else:
    root = Path.cwd()
    if root.name == "notebooks":
        os.chdir(root.parent)
    elif not (root / ".git").exists() and (root.parent / ".git").exists():
        os.chdir(root.parent)

ROOT = Path.cwd()
print("working directory:", ROOT)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", device)

if device == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("No GPU is visible. In Colab, switch the runtime type to GPU and reconnect.")

In [ ]:
from pathlib import Path
import shutil
import urllib.request

TINYSTORIES_VALID_URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt"
LECTURE_BYTES = 1_115_394

data_dir = ROOT / "data"
data_dir.mkdir(exist_ok=True)

raw_path = data_dir / "input.txt"
tiny_path = data_dir / "tiny-input"
lecture_path = ROOT / "input.txt"

if not raw_path.exists():
    print("downloading TinyStories validation split...")
    urllib.request.urlretrieve(TINYSTORIES_VALID_URL, raw_path)

text = raw_path.read_text(encoding="utf-8")
sep = "<|endoftext|>\n"

pieces = []
size = 0
chunks = text.split(sep)
for i, chunk in enumerate(chunks):
    piece = chunk if i == len(chunks) - 1 else chunk + sep
    piece_bytes = piece.encode("utf-8")
    if size + len(piece_bytes) > LECTURE_BYTES:
        break
    pieces.append(piece)
    size += len(piece_bytes)

if size < LECTURE_BYTES:
    pieces.append("\n" * (LECTURE_BYTES - size))

tiny_bytes = "".join(pieces).encode("utf-8")
assert len(tiny_bytes) == LECTURE_BYTES
tiny_bytes.decode("utf-8")

tiny_path.write_bytes(tiny_bytes)
shutil.copyfile(tiny_path, lecture_path)

print("wrote:", tiny_path, tiny_path.stat().st_size, "bytes")
print("wrote:", lecture_path, lecture_path.stat().st_size, "bytes")

## Lecture Starting Point

From here, continue with the lecture code. The only difference is that `input.txt` now contains lecture-sized TinyStories text.

In [ ]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("length of dataset in characters:", len(text))
print(text[:500])

In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)

In [ ]:
# create a mapping from characters to integers
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

print(encode("hello"))
print(decode(encode("hello")))

For a fast first Colab run, a roughly 1M-parameter version is easier to iterate on than the full lecture defaults. Use these values when you reach the hyperparameter cell:

In [ ]:
# Suggested toy settings for TinyStories on Colab.
# Replace the lecture hyperparameters with these for the first run.
batch_size = 64
block_size = 128
max_iters = 2000
eval_interval = 200
learning_rate = 3e-4
eval_iters = 50
n_embd = 96
n_head = 4
n_layer = 4
dropout = 0.2
device = "cuda" if torch.cuda.is_available() else "cpu"